# Notebook 4: Sistema RAG Clínico Multifuente (Jufe, Stahl y CANMAT) con Embeddings y Ollama

Este notebook implementa el sistema **RAG (Retrieval-Augmented Generation)** integrando el conocimiento de las tres fuentes clínicas especializadas:
1. **Dra. Gabriela Jufe** (*Psicofarmacología Práctica*): Posología, poblaciones especiales y manejo de efectos adversos.
2. **Dr. Stephen Stahl** (*Prescriber's Guide*): Perfil de receptores, interacciones enzimáticas por citocromos (CYP450) y efectos adversos potencialmente letales.
3. **Guías CANMAT / ISBD**: Algoritmos de 1ª/2ª línea, jerarquías de evidencia y prevención de cascadas/virajes.

Además, incorpora el catálogo de **Marcas Comerciales, Laboratorios y Códigos ATC de la OMS** para responder a consultas del mundo real.

---

### Arquitectura del RAG:
```
┌─────────────────────────────────────────────────────────────┐
│ 1. Consolidación de Datasets Limpios (Jufe + Stahl + CANMAT)│
└──────────────────────────────┬──────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│ 2. Construcción de Fichas Clínicas Estructuradas (Chunks)   │
└──────────────────────────────┬──────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│ 3. Indexación Vectorial (Embeddings con Ollama / Scikit)    │
└──────────────────────────────┬──────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│ 4. Retrieval Semántico (Top-K por Similitud de Coseno)      │
└──────────────────────────────┬──────────────────────────────┘
                               │
                               ▼
┌─────────────────────────────────────────────────────────────┐
│ 5. Augmented Generation con LLM en Ollama (Llama 3/Mistral) │
└─────────────────────────────────────────────────────────────┘
```


In [23]:
import os
import sys
import json
import re
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Detección dinámica y robusta de rutas
current_dir = Path.cwd()
project_root = current_dir if (current_dir / 'data').exists() else current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

DATA_PROCESSED = project_root / 'data' / 'processed'
DIR_PRETRAINING = project_root / 'pre_training'
print(f'Entorno RAG configurado.\n- Raíz: {project_root}\n- Datos Procesados: {DATA_PROCESSED}\n- Corpus pre_training: {DIR_PRETRAINING}')


Entorno configurado. Directorio de datos: /Users/anaisabecolinaarismendi/Documents/GitHub/Sofia/data/processed


## 1. Carga de Datasets Procesados y del Corpus Clínico `pre_training/`

Para maximizar la precisión diagnóstica y terapéutica, el motor RAG ahora ingesta dos niveles de información:
1. **Datasets Tabulares Procesados:** CSVs limpios de Jufe, Stahl y CANMAT para consulta relacional.
2. **Corpus Clínico Estructurado `pre_training/`:** 26 monografías narrativas, guías CANMAT, interacciones CYP450 y protocolos de desprescripción con metadatos YAML y resúmenes verbalizables (*Voice Snippets*).


In [24]:
# 1. Cargar Datasets Tabulares de Soporte
df_jufe = pd.read_csv(DATA_PROCESSED / 'jufe_farmacos_clean.csv')
df_jufe_efectos = pd.read_csv(DATA_PROCESSED / 'jufe_efectos_adversos_clean.csv')
df_stahl = pd.read_csv(DATA_PROCESSED / 'stahl_farmacos_clean.csv')
df_stahl_eventos = pd.read_csv(DATA_PROCESSED / 'stahl_eventos_adversos_clean.csv')
df_canmat = pd.read_csv(DATA_PROCESSED / 'canmat_recomendaciones_clean.csv')

# 2. Función Lectora del Corpus pre_training/
def cargar_corpus_pretraining(directorio: Path):
    docs = []
    if not directorio.exists():
        print(f'Aviso: Directorio {directorio} no encontrado.')
        return docs
        
    for f in sorted(directorio.glob('**/*.md')):
        if f.name.startswith('00_'):
            continue  # Omitir el manifiesto técnico
        with open(f, 'r', encoding='utf-8') as arch:
            texto = arch.read()
        
        parts = texto.split('---', 2)
        meta = {}
        cuerpo = texto
        if len(parts) >= 3:
            for line in parts[1].strip().splitlines():
                if ':' in line:
                    k, v = line.split(':', 1)
                    meta[k.strip()] = v.strip().strip('"').strip("'")
            cuerpo = parts[2].strip()
            
        voice_snippet = ''
        match_voice = re.search(r'## 1\. Resumen Ejecutivo Verbalizable.*?\n>(.*?)(?=\n##|\Z)', cuerpo, re.DOTALL)
        if match_voice:
            voice_snippet = match_voice.group(1).replace('\n>', ' ').replace('"', '').strip()
            
        docs.append({
            'id': meta.get('id', f.stem),
            'tipo': meta.get('tipo', f.parent.name),
            'titulo': meta.get('titulo', meta.get('farmaco', f.stem.replace('_', ' ').title())),
            'atc': meta.get('atc_codigo', 'N/D'),
            'voice_snippet': voice_snippet,
            'contenido': cuerpo,
            'archivo_origen': str(f.relative_to(project_root))
        })
    return docs

docs_pretraining = cargar_corpus_pretraining(DIR_PRETRAINING)
print(f'Datos cargados exitosamente:')
print(f'- Tablas CSV: Jufe ({len(df_jufe)} fármacos) | Stahl ({len(df_stahl)} fármacos) | CANMAT ({len(df_canmat)} directrices)')
print(f'- Corpus pre_training: {len(docs_pretraining)} documentos Markdown ricos parseados.')


Jufe: 5 fármacos | Stahl: 4 fármacos | CANMAT: 19 directrices.


## 2. Enriquecimiento con Marcas Comerciales y Laboratorios

Para cumplir el requisito de consultar por **marca comercial** y **laboratorio fabricante**, creamos una base de homologación clínica de las marcas más prescritas en psiquiatría:


In [25]:
CATALOGO_MARCAS = pd.DataFrame([
    {"farmaco": "Sertralina", "marca": "Zoloft", "laboratorio": "Pfizer", "pais": "Internacional", "presentacion": "Comprimidos 50mg / 100mg"},
    {"farmaco": "Sertralina", "marca": "Serlift", "laboratorio": "Roemmers", "pais": "Argentina/LatAm", "presentacion": "Comprimidos 50mg / 100mg"},
    {"farmaco": "Sertralina", "marca": "Aremis", "laboratorio": "Almirall", "pais": "España", "presentacion": "Comprimidos 50mg / 100mg"},
    {"farmaco": "Fluoxetina", "marca": "Prozac", "laboratorio": "Eli Lilly", "pais": "Internacional", "presentacion": "Cápsulas 20mg"},
    {"farmaco": "Fluoxetina", "marca": "Foxetin", "laboratorio": "Gador", "pais": "Argentina/LatAm", "presentacion": "Comprimidos 20mg"},
    {"farmaco": "Paroxetina", "marca": "Paxil / Seroxat", "laboratorio": "GlaxoSmithKline (GSK)", "pais": "Internacional", "presentacion": "Comprimidos 20mg"},
    {"farmaco": "Paroxetina", "marca": "Aropax", "laboratorio": "GSK", "pais": "LatAm", "presentacion": "Comprimidos 20mg"},
    {"farmaco": "Paroxetina", "marca": "Psicoasten", "laboratorio": "Baliarda", "pais": "Argentina", "presentacion": "Comprimidos 20mg"},
    {"farmaco": "Clozapina", "marca": "Clozaril", "laboratorio": "Novartis", "pais": "Internacional", "presentacion": "Comprimidos 25mg / 100mg"},
    {"farmaco": "Clozapina", "marca": "Lapenax", "laboratorio": "Novartis", "pais": "LatAm", "presentacion": "Comprimidos 25mg / 100mg"},
    {"farmaco": "Risperidona", "marca": "Risperdal", "laboratorio": "Janssen-Cilag", "pais": "Internacional", "presentacion": "Comprimidos 1mg / 2mg / 3mg"},
    {"farmaco": "Risperidona", "marca": "Sequinan", "laboratorio": "Roemmers", "pais": "Argentina", "presentacion": "Comprimidos 1mg / 2mg / 3mg"},
    {"farmaco": "Haloperidol", "marca": "Haldol", "laboratorio": "Janssen-Cilag", "pais": "Internacional", "presentacion": "Gotas / Comprimidos 5mg / Ampollas"},
    {"farmaco": "Litio (Carbonato de Litio)", "marca": "Plenur", "laboratorio": "Faes Farma", "pais": "España", "presentacion": "Comprimidos 400mg"},
    {"farmaco": "Litio (Carbonato de Litio)", "marca": "Ceglution", "laboratorio": "Baliarda", "pais": "Argentina", "presentacion": "Comprimidos 300mg"},
    {"farmaco": "Bupropión", "marca": "Wellbutrin / Elontril", "laboratorio": "GSK", "pais": "Internacional", "presentacion": "Comprimidos liberación prolongada 150mg / 300mg"},
    {"farmaco": "Lamotrigina", "marca": "Lamictal", "laboratorio": "GSK", "pais": "Internacional", "presentacion": "Comprimidos dispersables 25mg / 50mg / 100mg"},
    {"farmaco": "Lamotrigina", "marca": "Lamotrax", "laboratorio": "Montpellier", "pais": "Argentina", "presentacion": "Comprimidos 25mg / 50mg / 100mg"}
])

print(f"Catálogo de marcas registrado: {len(CATALOGO_MARCAS)} marcas comerciales con laboratorios.")
CATALOGO_MARCAS.head(5)


Catálogo de marcas registrado: 18 marcas comerciales con laboratorios.


,farmaco,marca,laboratorio,pais,presentacion
0,Sertralina,Zoloft,Pfizer,Internacional,Comprimidos 50mg / 100mg
1,Sertralina,Serlift,Roemmers,Argentina/LatAm,Comprimidos 50mg / 100mg
2,Sertralina,Aremis,Almirall,España,Comprimidos 50mg / 100mg
3,Fluoxetina,Prozac,Eli Lilly,Internacional,Cápsulas 20mg
4,Fluoxetina,Foxetin,Gador,Argentina/LatAm,Comprimidos 20mg


## 3. Construcción y Fusión de la Base de Conocimiento RAG

Integramos los 26 documentos de alta densidad semántica provenientes de `pre_training/` junto con las directrices específicas de CANMAT y el catálogo de marcas comerciales.


In [26]:
documentos_rag = []

# 1. Incorporar los documentos curados del corpus pre_training/ (Monografías, CYP450, Cascadas, CANMAT, Alertas)
documentos_rag.extend(docs_pretraining)

# 2. Añadir fichas directas de guías CANMAT (por si se requiere granularidad por fármaco individual)
for _, row_c in df_canmat.iterrows():
    doc_id = f"canmat_{row_c['patologia'][:3].lower()}_{row_c['farmaco'].split()[0].lower()}"
    # Evitar duplicados si ya existe un documento más completo
    if not any(d['id'] == doc_id for d in documentos_rag):
        contenido_guia = f"""# GUÍA CLÍNICA CANMAT: {row_c['patologia'].upper()} ({row_c['fase'].upper()})
- Recomendación: {row_c['linea']}
- Estrategia: {row_c['estrategia']}
- Fármaco / Esquema: {row_c['farmaco']} (ATC: {row_c['atc']})
- Nivel de Evidencia Científica: {row_c['nivel_evidencia']}
- Directriz Clínica: {row_c['comentarios']}
- Fuente: Canadian Network for Mood and Anxiety Treatments (CANMAT)
"""
        documentos_rag.append({
            'id': doc_id,
            'tipo': 'guia_canmat',
            'titulo': f"CANMAT: {row_c['patologia']} - {row_c['farmaco']}",
            'atc': row_c['atc'],
            'voice_snippet': f"CANMAT recomienda {row_c['farmaco']} en {row_c['linea']} para {row_c['patologia']} ({row_c['fase']}) con evidencia {row_c['nivel_evidencia']}.",
            'contenido': contenido_guia.strip(),
            'archivo_origen': 'data/processed/canmat_recomendaciones_clean.csv'
        })

print(f'Total de documentos clínicos indexables en el RAG: {len(documentos_rag)}')


Total de documentos indexables en el RAG: 28


## 4. Motor de Embeddings y Almacén Vectorial

Diseñamos una clase `VectorStore` híbrida:
* **Conexión nativa a Ollama:** Intenta generar embeddings mediante el endpoint local `http://localhost:11434/api/embeddings` (modelo `nomic-embed-text` o `bge-m3`).
* **Fallback determinista con Scikit-Learn (TF-IDF Semántico):** Si Ollama no está iniciado en segundo plano, utiliza un vectorizador denso local para que el notebook funcione de inmediato.


In [27]:
class ClinicalVectorStore:
    def __init__(self, ollama_host='http://localhost:11434', embedding_model='embeddinggemma:latest'):
        self.ollama_host = ollama_host
        self.embedding_model = embedding_model
        self.documentos = []
        self.embeddings = None
        self.use_ollama = False
        self._verificar_ollama()
        
    def _verificar_ollama(self):
        try:
            r = requests.get(f'{self.ollama_host}/api/tags', timeout=2)
            if r.status_code == 200:
                self.use_ollama = True
                print(f'✓ Conectado a Ollama local ({self.ollama_host}).')
            else:
                self.use_ollama = False
        except Exception:
            self.use_ollama = False
            print('ℹ Ollama no está activo. Se usará el vectorizador TF-IDF local de Scikit-Learn.')

    def indexar_documentos(self, docs):
        self.documentos = docs
        textos = [f"{d['titulo']} {d.get('atc', '')}\n{d['contenido']}" for d in docs]
        
        if self.use_ollama:
            try:
                print(f'Generando embeddings con Ollama ({self.embedding_model}) para {len(docs)} documentos...')
                emb_list = []
                for t in textos:
                    res = requests.post(
                        f'{self.ollama_host}/api/embeddings',
                        json={'model': self.embedding_model, 'prompt': t[:2000]},
                        timeout=15
                    )
                    emb_list.append(res.json()['embedding'])
                self.embeddings = np.array(emb_list)
                print(f'✓ Embeddings calculados exitosamente con Ollama ({self.embedding_model}).')
                return
            except Exception as e:
                print(f'Error generando embeddings en Ollama: {e}. Conmutando a fallback local.')
                self.use_ollama = False
                
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=1500)
        self.embeddings = self.vectorizer.fit_transform(textos).toarray()
        print(f'✓ Indexados {len(self.documentos)} documentos con vectorizador semántico local.')

    def buscar(self, query, top_k=2):
        if self.use_ollama:
            try:
                res = requests.post(
                    f'{self.ollama_host}/api/embeddings',
                    json={'model': self.embedding_model, 'prompt': query},
                    timeout=5
                )
                query_vec = np.array(res.json()['embedding']).reshape(1, -1)
                sims = cosine_similarity(query_vec, self.embeddings)[0]
            except Exception:
                query_vec = self.vectorizer.transform([query]).toarray()
                sims = cosine_similarity(query_vec, self.embeddings)[0]
        else:
            query_vec = self.vectorizer.transform([query]).toarray()
            sims = cosine_similarity(query_vec, self.embeddings)[0]
            
        indices_top = np.argsort(sims)[::-1][:top_k]
        
        resultados = []
        for idx in indices_top:
            doc = self.documentos[idx].copy()
            doc['score_similitud'] = round(float(sims[idx]), 4)
            resultados.append(doc)
            
        return resultados

# Inicializar e indexar la base unificada
vector_store = ClinicalVectorStore()
vector_store.indexar_documentos(documentos_rag)


✓ Conectado a Ollama local (http://localhost:11434).
Embeddings calculados con Ollama (embeddinggemma:latest).


## 5. Módulo de Generación Asistida con el Nuevo 'Soul' de Sofia (RAG Prompting & Ollama)

El 'Soul' de Sofia ha sido optimizado con técnicas de IA para incorporar:
- **Identidad Profesional:** Médica especialista en psicofarmacología clínica y farmacovigilancia.
- **Grounding Estricto y Citaciones:** Obligación de citar a Jufe, Stahl o CANMAT sin alucinaciones.
- **Alerta Activa de Farmacovigilancia:** Detección de cascadas de prescripción e interacciones enzimáticas CYP450.
- **Formato Médico Estructurado en 4 Secciones:** Juicio Clínico Inmediato, Mecanismo/Cinética, Pauta Operativa y Alertas.


In [28]:
PROMPT_SISTEMA_CLINICO = """Eres Sofia, médica especialista en Psicofarmacología Clínica, Psiquiatría Basada en la Evidencia y Farmacovigilancia.
Tu función es asesorar a médicos adjuntos, residentes y profesionales de la salud con el máximo rigor científico, basándote EXCLUSIVAMENTE en el CONTEXTO CLÍNICO RECUPERADO.

### DIRECTRICES DE RAZONAMIENTO Y SEGURIDAD CLÍNICA:
1. GROUNDING ESTRICTO: Basa todas tus afirmaciones en los documentos del contexto recuperado (Dra. Gabriela Jufe, Dr. Stephen Stahl, Guías oficiales CANMAT/ISBD y catálogo de marcas oficiales). Si el contexto no contiene datos suficientes para responder con seguridad, declara explícitamente la limitación sin inventar datos.
2. CITACIÓN DE FUENTES: Identifica de qué fuente proviene cada recomendación clínica (ej: 'Según la Dra. Gabriela Jufe...', 'El Dr. Stephen Stahl describe...', 'De acuerdo con las guías CANMAT (Evidencia Level 1)...').
3. FARMACOVIGILANCIA Y DETECCIÓN DE CASCADAS: Si la consulta plantea el uso de un fármaco para tratar el efecto adverso de otro (ej. haloperidol con biperideno), resalta de forma prioritaria una 'ALERTA DE CASCADA DE PRESCRIPCIÓN' y orienta hacia la desescalada o rotación a un fármaco con menor perfil adverso.
4. METABOLISMO ENZIMÁTICO Y CITOCROMOS: Especifica siempre las vías metabólicas y los citocromos involucrados (CYP1A2, CYP2D6, CYP3A4, CYP2B6, glucuronidación UGT1A4 o eliminación renal de litio).

### FORMATO DE RESPUESTA MÉDICA OBLIGATORIO:
Estructura tu respuesta en los siguientes 4 apartados numerados:
- **1. Juicio Clínico Inmediato:** Conclusión directa, asertiva y aplicable a la consulta en 2-3 líneas.
- **2. Mecanismo Farmacológico y Cinética:** Explicación fisiopatológica, receptores involucrados (D2, 5-HT, etc.) y/o perfil enzimático CYP450.
- **3. Pauta Operativa y Recomendaciones:** Dosis sugeridas, velocidad de titulación/desescalada y parámetros obligatorios de monitorización analítica o clínica.
- **4. Alertas de Seguridad y Trazabilidad Comercial:** Advertencias críticas (caja negra, litemia, agranulocitosis) y nombres comerciales / laboratorios aplicables.
"""

def generar_respuesta_rag(pregunta, vector_store, modelo_ollama="qwen3.5:2b-mlx", top_k=2):
    docs_recuperados = vector_store.buscar(pregunta, top_k=top_k)
    
    contexto_str = "\n\n---\n\n".join([
        f"### [{d['titulo']}] (Tipo: {d.get('tipo', 'clinico')} | ATC: {d.get('atc', 'N/D')})\n{d['contenido']}"
        for d in docs_recuperados
    ])
    
    prompt_completo = f"""{PROMPT_SISTEMA_CLINICO}

==================================================
CONTEXTO CLÍNICO RECUPERADO:
{contexto_str}
==================================================
PREGUNTA DEL MÉDICO / USUARIO:
{pregunta}
==================================================
RESPUESTA CLÍNICA DE SOFIA:"""

    if vector_store.use_ollama:
        try:
            res = requests.post(
                f"{vector_store.ollama_host}/api/generate",
                json={
                    "model": modelo_ollama,
                    "prompt": prompt_completo,
                    "stream": False,
                    "options": {
                        "temperature": 0.2,
                        "top_p": 0.85,
                        "num_predict": 500
                    }
                },
                timeout=120
            )
            return res.json()["response"], docs_recuperados
        except Exception as e:
            print(f"Aviso: Error con Ollama ({e}). Mostrando síntesis clínica local.")

    # Fallback determinista local
    sintesis = f"**1. Juicio Clínico Inmediato:** Para la consulta sobre '{pregunta}', se recuperaron documentos clave:\n"
    for d in docs_recuperados:
        sintesis += f"- {d['titulo']} (Relevancia: {d['score_similitud']})\n"
        if d.get('voice_snippet'):
            sintesis += f"  > Resumen Verbal: \"{d['voice_snippet'][:180]}...\"\n"
    return sintesis, docs_recuperados


## 6. Pruebas y Casos de Uso del RAG

A continuación, ejecutamos las 4 consultas clave del proyecto final para verificar la recuperación semántica y la respuesta médica:

---

### Caso 1: Principio activo, marcas comerciales y laboratorios


In [29]:
pregunta_1 = "¿Qué laboratorios fabrican la Sertralina y cuáles son sus nombres comerciales y posología según Jufe?"
respuesta, fuentes = generar_respuesta_rag(pregunta_1, vector_store)
print(respuesta)


Según Dra. Gabriela Jufe en su ficha farmacológica sobre la Sertralina (N06AB06):

*   **Laboratorios que fabrican Sertralina:** Los laboratorios correspondientes a las marcas de la sertralina son:
    *   Zoloft (Pfizer)
    *   Serlift (Roemmers)
    *   Aremis (Almirall)

*   **Posología según Jufe:** La posología se establece en los siguientes rangos:
    *   **Posición inicial:** 25-50 mg/día.
    *   **Rango terapéutico:** 50-200 mg/día.

La guía CANMAT indica que la Sertralina (ATC: N06AB06) es la primera línea para el tratamiento del Trastorno de Depresivo Mayor (TDM) y sugiere monoterapia, pero solo proporciona los nombres comerciales y posología según la información recuperada en Jufe. La guía CANMAT también indica que Sertralina está segura en pacientes con cardiopatía isquémica.


### 📋 Respuesta Clínica Registrada (Caso 1)
> **Documento Recuperado:** `Sertralina` (Similitud: `0.666`)  
> **Fuente Médica:** Dra. Gabriela Jufe (*Psicofarmacología Práctica*)

**Respuesta Clínica**

Basado en la ficha farmacológica y la información clínica proporcionada:

1.  **Laboratorios y Marcas Comerciales:**
    *   **Pfizer:** Fabrica el fármaco bajo la marca comercial **Zoloft**.
    *   **Roemmers:** Fabrica el fármaco bajo la marca comercial **Serlift**.
    *   **Almirall:** Fabrica el fármaco bajo la marca comercial **Aremis**.

2.  **Posología (Dosis de Administración):**
    *   **Dosis Inicial (Inicio):** 25-50 mg/día.
    *   **Rango Terapéutico:** 50-200 mg/día.

**Fuente Médica:** Dra. Gabriela Jufe - Psicofarmacología Práctica


### Caso 2: Interacciones enzimáticas de citocromos (CYP450) según Stahl


In [34]:
pregunta_2 = "¿Qué interacciones y precauciones enzimáticas con CYP2D6 describe Stahl para Paroxetina y Bupropión?"
respuesta, fuentes = generar_respuesta_rag(pregunta_2, vector_store, modelo_ollama="qwen3.5:2b-mlx")
print(respuesta)


A continuación presento la respuesta estructurada según el análisis de las fichas farmacológicas y el perfil del Dr. Stephen Stahl proporcionado en el contexto clínico recuperado:

### 1. Interacción y Metabolismo Citocromo CYP2D6 (Estilo Stahl + Context)

Según **Dr. Stephen Stahl (Guía del Prescriptor)** y la ficha farmacológica recuperada, la interacción entre el paroxetina y el bupropión con respecto al enzimamiento citocromo es crítica debido a la potencia de inhibición o sustrato:

*   **Paroxetina (Paxil/Seroxat):** Se describe que esta medicación es un "Potente inhibitor del CYP2D6" y sustrato fuerte. Su metabolismo se ve afectado por la competencia con enzimas del citocromo en el metabolismo.
*   **Bupropión (Wellbutrin/Elontril):** El contexto indica que este fármaco se "Metabolizado por CYP2B6" y es también un "Potente inhibidor de CYP2D6". La combinación de estos dos agentes, ambos con potente acción sobre el CYP450, implica un riesgo de interacción farmacocinética signific

### 📋 Respuesta Clínica Registrada (Caso 2)
> **Documento Recuperado:** `Bupropión` (Similitud: `0.558`)  
> **Fuente Médica:** Dr. Stephen Stahl (*Prescriber's Guide*)

Basado en la información clínica provista y el perfil de Stahl (Clase IRND) para Bupropión, junto con las indicaciones estándar de Farmacología Clínica derivadas del Jefe y Canmat (CANMAT), aquí se presenta el análisis:

**RESPUESTA CLÍNICA:**

Sí existe una interacción relevante entre Paroxetina (Paroxetina) y Bupropión debido a la acción enzimática compartida sobre las enzimas citocromo CYP. Aunque Stahl no detalla específicamente la co-efecto "Paroxetin + Bupropion", sus manuales de prescripción señalan la precaución por el metabolismo enzimático de ambas fármacos.

### 1. Interacciones y Precauciones enzimáticas según Stahl:
El perfil especializado del **Stahl** indica que ambos medicamentos son metabolizados por enzimas del citocromo P450 (CYP). La interacción principal se basa en el hecho de que **Bupropión es un inhibitoro potente de CYP2D6**, mientras que Paroxetina depende en gran parte de este sistema para su descomposición.

### 2. Precaución:
*   **Riesgo:** Con la co-administración, el riesgo de **acumulación** de Paroxetina aumenta debido a la inhibición de CYP2D6 por Bupropión, mientras que el Bupropión puede aumentar su toxicidad si se metaboliza por vías normales sin la descomposición.
*   **Estimación:** En casos clínicos donde ambos están prescritos, es necesario monitorear los niveles de paroxetina y asegurar una adecuada separación del tiempo (generalmente más de 1 semana) o ajustar dosis si hay historia de abuso de alcohol/medicamentos, siguiendo las advertencias de Jefe en la clínica.

### 3. Referencia:
*   **Stahl:** *"Bupropión es un potente inhibidor de CYP2D6"*.
*   **Jefe/Stahl:** La interacción debe ser evaluada con el cuidado debido a la posible alteración de la farmacocinética de Paroxetina.

---
**Nota Clínica:**
Se recomienda consultar siempre los manuales de Stahl y las recomendaciones del Jefe sobre la co-administración de antidepresivos y iróns (Bupropión) para evitar sobredosis o eventos adversos graves, especialmente en pacientes con antecedentes de epilepsia (risco convulsivo) y paroxetina.


### Caso 3: Guías CANMAT - Tratamiento y Contraindicación de Viraje


In [31]:
pregunta_3 = "¿Qué fármacos recomienda CANMAT en primera línea para manía aguda y por qué desaconseja los antidepresivos solos en depresión bipolar?"
respuesta, fuentes = generar_respuesta_rag(pregunta_3, vector_store)
print(respuesta)


Respuesta Clínica Estructurada

**1. Farmacoterapia Prima de LÍnea para Manía Aguda (CanMAT)**

Según la **Guía Clínica CANMAT**, para el trastorno bipolar con manía aguda y crisis de manía, la primera línea recomendada es:

*   **Estrategia:** Combinación de fármacos.
*   **Fármaco / Esquema:** Litio + Antipsicótico Atípico (ATC: N/D).
*   **Nivel de Evidencia Científica:** Nivel 1.
*   **Directriz Clínica:** Mayor tasa de respuesta en manía severa que la monoterapia (CanMAT indica esto como Directriz Clínica en Manía Aguda).

> **Fuente Referida:** CanMAT, *Guía Clínica CANMAT*.
>
> **Notas Clínicas:**
> Este esquema busca una alta estabilidad clínica y respuesta rápida en casos de manía severa ("MANÍA AGUDA"). La guía considera esta combinación superior a otras monoterapias, asegurando un manejo seguro que evita el riesgo de depresión o suicidio asociado con el antidepresivo en sí mismo.

**2. Contraindicación para Antidepressivos (Monoterapia) en Trastorno Bipolar (Depresión)**

En

### 📋 Respuesta Clínica Registrada (Caso 3)
> **Documento Recuperado:** `CANMAT: Trastorno Bipolar - Litio + Antipsicótico Atípico` (Similitud: `0.7282`)  
> **Fuente Médica:** Guías CANMAT / ISBD (*Trastornos del Ánimo*)

**Respuesta Clínica Directa:**

Según la **Guía Clínica CANMAT**, el tratamiento de primera línea para **manía aguda** en trastorno bipolar implica la combinación de dos fármacos clave: **Litio + Antipsicótico Atípico**.

La estrategia recomendada se fundamenta en una evidencia científica con **nivel 1 (Level 1)**, que establece claramente que esta estrategia ofrece mayor tasa de respuesta en casos severos de manía comparada con la monoterapia.

### Por qué no se desaconsejan los antidepresivos solos en depresión bipolar:

El canonamiento de CANMAT en su Directriz Clínica indica explícitamente que **desaconseja los antidepresivos solos (monoterapia)** para el tratamiento de la depresión bipolar. La razón principal es la alta tasa de riesgo asociado al uso de estos fármacos sin antipsicotico o litio en pacientes con trastorno bipolar, lo que podría llevar a:
1.  **Inducción de manía:** Uso incorrecto del antidepresivo como tratamiento primario puede precipitar el inicio de manía o ciclos recurrentes más graves.
2.  **Desregulación del ciclo:** En pacientes con depresión y bipolaridad, el uso aislado de estos fármacos es generalmente ineficaz para mantener la estabilidad emocional en comparación con combinaciones complejas que incluyen litio.

*Fuente:* Canadian Network for Mood and Anxiety Treatments (CANMAT), *Guía Clínica*, Nivel 1, Estrategia Combinada (Litio + Antipsicótico).


### Caso 4: Detección de Cascada de Prescripción (Haloperidol y Biperideno)


In [32]:
pregunta_4 = "Un paciente en tratamiento con Haloperidol comienza con rigidez y temblor y le indican medicación antiparkinsoniana. ¿Qué dicen Jufe y Stahl sobre este riesgo?"
respuesta, fuentes = generar_respuesta_rag(pregunta_4, vector_store)
print(respuesta)


**Respuesta Clínica Estructurada:**

Según la ficha clínica proporcionada por **Dra. Gabriela Jufe (Psicofarmacología Práctica)**, los riesgos asociados con la co-prescripción del Haloperidol y antiparkinsonianos presentan limitaciones específicas que deben ser revisadas en el contexto clínico:

### 1. Riesgo de Parkinsonismo Secundario
**Fuente:** Dra. Gabriela Jufe (Haloperidol).
> La información indica que el Haloperidol, como un antipsicótico típico, está asociado con efectos adversos frecuentes como **"Parkinsonismo secundario"**. El documento establece una posología usual de **2-10 mg/día** en fase aguda.

*Interpretación:* Dado que el paciente presenta rigidez y temblor, es probable que esto represente un efecto parásico o una exacerbación del antipsicótico sobre los receptores dopaminérgicos. La combinación con medicación antiparkinsoniana debe vigilarse para no agravar la secuencia motora, ya que el Haloperidol tiene riesgo de confusión con Parkinson idiopático (según Jufe).



### 📋 Respuesta Clínica Registrada (Caso 4)
> **Documento Recuperado:** `Haloperidol` (Similitud: `0.5781`)  
> **Fuente Médica:** Jufe & Stahl (*Riesgo de Cascada de Prescripción*)

**RESPUESTA CLÍNICA:**

Según la información proporcionada por **Dra. Gabriela Jufe** (Jufe), existe un **alto riesgo de inducir cascadas de medicación en geriatría**. Este efecto es debido a la **confusión entre el Síndrome del Parkinson Idiopático y las SEPs** (Serotonergias o interacciones complejas) con los antipsicóticos.

En el contexto general de farmacología, **Stahl (Stahl)** confirma que cuando se unen antipsicóticos típicos (como Haloperidol) con medicamentos antiparkinsonianos en ancianos, se aumenta drásticamente el riesgo de complicaciones por confusión diagnóstica.

Citas médicas:
*   **Jufe:** "Poblaciones Especiales: Alto riesgo de inducir cascadas de medicación en geriatría por confusión de SEP con Parkinson idiopático." (Dra. Gabriela Jufe - Psicofarmacología Práctica).
*   **Stahl:** Confirma que la interacción entre antipsicóticos y fármacos para Parkinson en adultos mayores genera una alta probabilidad de complicaciones farmacológicas debido a los efectos secundarios acumulados (Parkinsonismo, discinesia tardía) y riesgo de confusiones diagnósticas.


## 7. Exportación de la Base de Conocimiento a JSON / SQLite

Guardamos las fichas clínicas integradas en `data/processed/rag_knowledge_base.json` para que puedan ser cargadas directamente por tu aplicación de **Streamlit**.


In [33]:
out_rag_json = DATA_PROCESSED / "rag_knowledge_base.json"

with open(out_rag_json, "w", encoding="utf-8") as f:
    json.dump(documentos_rag, f, indent=2, ensure_ascii=False)

print(f"Base de conocimiento RAG exportada con éxito ({len(documentos_rag)} documentos) en: {out_rag_json}")


Base de conocimiento RAG exportada con éxito (28 documentos) en: /Users/anaisabecolinaarismendi/Documents/GitHub/Sofia/data/processed/rag_knowledge_base.json
